In [1]:
import csv
import random
import math

In [5]:
def loadcsv(filename):
    lines = csv.reader(open(filename, "r"))
    # Skip the header row
    next(lines)
    dataset = list(lines)
    for i in range(len(dataset)):
        # Convert strings into numbers
        dataset[i] = [float(x) for x in dataset[i]]
    return dataset

In [26]:
def splitdataset(dataset, splitratio):
  # 67% training size
  trainsize = int(len(dataset) * splitratio)
  trainset = []
  copy = list(dataset)
  while len(trainset) < trainsize:
    # Generate random index
    index = random.randrange(len(copy))
    trainset.append(copy.pop(index))
  return [trainset, copy]

In [7]:
def separatebyclass(dataset):
  separated = {}
  # Separate data according to class
  for i in range(len(dataset)):
    vector = dataset[i]
    if vector[-1] not in separated:
      separated[vector[-1]] = []
      separated[vector[-1]].append(vector)
  return separated

In [8]:
def mean(numbers):
  return sum(numbers) / float(len(numbers))

In [31]:
def stdev(numbers):
  if len(numbers) <= 1:
    return 0.0 # Standard deviation is 0 for a single point or empty list
  avg = mean(numbers)
  variance = sum(
      [pow(x - avg, 2) for x in numbers]
      ) / float(len(numbers) - 1)
  return math.sqrt(variance)

In [12]:
def summarize(dataset):
  summaries = [
      (mean(attribute), stdev(attribute))
      for attribute in zip(*dataset)
  ]
  del summaries[-1]
  return summaries

In [27]:
def summarizebyclass(dataset):
  separated = separatebyclass(dataset)
  summaries = {}
  for classvalue, instances in separated.items():
     # Calculate mean and standard deviation
     summaries[classvalue] = summarize(instances)
  return summaries

In [35]:
def calculateprobability(x, mean, stdev):
  if stdev == 0:
    return 1.0 if x == mean else 0.0 # If stdev is 0, probability is 1 if x is mean, else 0
  exponent = math.exp(
      -(math.pow(x - mean, 2) /
       (2 * math.pow(stdev, 2)))
  )
  return (
      (1 / (math.sqrt(2 * math.pi) * stdev))
      * exponent
  )

In [36]:
def calculateclassprobabilities(summaries, inputvector):
  probabilities = {}
  for classvalue, classsummaries in summaries.items():
    probabilities[classvalue] = 1
    for i in range(len(classsummaries)):
      mean_value, stdev_value = classsummaries[i]
      x = inputvector[i]
      probabilities[classvalue] *= calculateprobability(
          x,
          mean_value,
          stdev_value
      )
  return probabilities

In [15]:
def predict(summaries, inputvector):
   probabilities = calculateclassprobabilities(
       summaries,
       inputvector
    )
   bestLabel = None
   bestProb = -1
   for classvalue, probability in probabilities.items():
    if bestLabel is None or probability > bestProb:
      bestProb = probability
      bestLabel = classvalue
   return bestLabel

In [16]:
def getpredictions(summaries, testset):
  predictions = []
  for i in range(len(testset)):
    result = predict(
        summaries,
        testset[i]
    )
    predictions.append(result)
  return predictions

In [17]:
def getaccuracy(testset, predictions):
  correct = 0
  for i in range(len(testset)):
    if testset[i][-1] == predictions[i]:
      correct += 1
  return (correct / float(len(testset))) * 100.0

In [28]:
def main():
  # CSV file name
  filename = '/content/sample_data/mnist_train_small.csv'
  splitratio = 0.67
  dataset = loadcsv(filename)

  trainingset, testset = splitdataset(
      dataset,
      splitratio
  )
  print(
      'Split {0} rows into train={1} and test={2} rows'
       .format(
           len(dataset),
           len(trainingset),
           len(testset)
           )
  )
  # Calculate summaries for each class in the training set
  summaries = summarizebyclass(trainingset)

  # Generate predictions for the test set
  predictions = getpredictions(summaries, testset)

  # Calculate accuracy
  accuracy = getaccuracy(testset, predictions)
  print('Accuracy: {0}%'.format(accuracy))

In [38]:
if __name__ == "__main__":
   main()

Split 19999 rows into train=13399 and test=6600 rows
Accuracy: 100.0%
